In [0]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OrdinalEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)
from sklearn.metrics import precision_recall_curve
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

import matplotlib.pyplot as plt
import seaborn as sns

## Load raw data

In [0]:
df = pd.read_csv("/Workspace/Users/ajiboyeniola@gmail.com/lead-scoring/data/raw/leads.csv")

df.shape

## Drop unnecessary columns

In [0]:
df = pd.DataFrame(df)
df = df.drop(['lead_id', 'company_name', 'created_date', 'job_title', 'email_clicks', 'pages_viewed'], axis=1)

df.shape

## Train test split: Stratified Split

In [0]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = df.drop(columns=['converted']) # Creates the feature matrix X by dropping the target column "converted"
y = df['converted'] # Creates the target vector y by selecting only the target column "converted"

# Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2, # 20% of the data for testing
    stratify=y, # ensures the split preserves the same ratio of "converted" classes in both sets, which is important for class imbalance
    random_state=42 # Seeds the randomness so the split is reproducible every time i run it
)

# Confirm shapes
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

# Confirm stratification worked
print(f"\ny_train distribution:\n{y_train.value_counts(normalize=True)}")
print(f"\ny_test distribution:\n{y_test.value_counts(normalize=True)}")

## Handle missing values

In [0]:
missing_values = pd.isnull(df).sum()
missing_values

In [0]:
# After train test split: fill on X_train and X_test only

# Numerical — median from train only: calculates the median of that column from only training data
median_days = X_train['days_since_last_contact'].median()
X_train['days_since_last_contact'] = X_train['days_since_last_contact'].fillna(median_days)
X_test['days_since_last_contact'] = X_test['days_since_last_contact'].fillna(median_days)

median_reply = X_train['email_reply_rate'].median()
X_train['email_reply_rate'] = X_train['email_reply_rate'].fillna(median_reply)
X_test['email_reply_rate'] = X_test['email_reply_rate'].fillna(median_reply)

median_response = X_train['response_time_days'].median()
X_train['response_time_days'] = X_train['response_time_days'].fillna(median_response)
X_test['response_time_days'] = X_test['response_time_days'].fillna(median_response)

# Binary — fill with 0
X_train['budget_indicated'] = X_train['budget_indicated'].fillna(0)
X_test['budget_indicated'] = X_test['budget_indicated'].fillna(0)

X_train['pricing_page_visited'] = X_train['pricing_page_visited'].fillna(0)
X_test['pricing_page_visited'] = X_test['pricing_page_visited'].fillna(0)

# Categorical — fill with Unknown
X_train['ad_platform'] = X_train['ad_platform'].fillna('Unknown')
X_test['ad_platform'] = X_test['ad_platform'].fillna('Unknown')

# Confirm no missing values
print(X_train.isnull().sum()[X_train.isnull().sum() > 0])
print(X_test.isnull().sum()[X_test.isnull().sum() > 0])

## Build Column Transformer

In [0]:
# Numerical columns that will be scaled
robust_cols = ['lead_age_days', 'email_opens', 'website_visits', 'content_downloads', 'days_since_last_contact', 'ad_clicks', 'num_contacts', 'email_reply_rate', 'response_time_days',
    'interactions_last_30_days']

# Categorical columns with natural order
oe_cols = ['company_size', 'funnel_stage', 'preferred_contact_time']

# Catgegorical columns with no natural order
ohe_cols = ['business_type', 'industry', 'lead_source', 'ad_platform']

# Preprocessing pipeline using column transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('robust', RobustScaler(), robust_cols),
        ('oe', OrdinalEncoder(categories=[
            ['Small', 'Medium', 'Large'],
            ['Awareness', 'Consideration', 'Decision'],
            ['Morning', 'Afternoon', 'Evening']
        ]), oe_cols),
        ('ohe', OneHotEncoder(handle_unknown='ignore'), ohe_cols)
    ],
    remainder='passthrough' # Any columns not explicitly defined in the transformers list will be passed through unchanged rather than being dropped
)

### Calculate class imbalance and define the models, each configured to handle that imbalance

In [0]:
# Counts how many leads didn't convert (0) and how many did convert (1) in the training set (Used by XGBoost)
neg_count = (y_train == 0).sum() # Number of negative count
pos_count = (y_train == 1).sum() # Number of positive count

models = {
    "Logistic Regression": LogisticRegression(
        class_weight='balanced', # To handle imbalance class
        max_iter=10000, # logistic regression is an iterative process, and the default max_iter is 100. If the model doesn't converge within 100 iterations, it will throw an error. Increase the max_iter to a higher value to allow the model to converge.
        random_state=42 # Reproducibility
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, # Number of trees to build and combine their votes
        class_weight='balanced',
        random_state=42,
        n_jobs=-1 # Use all available CPU cores to train tress in parallel
    ),
    "XGBoost": XGBClassifier(
        scale_pos_weight = neg_count / pos_count, # To know the ratio of negative to positive class before defining the model, to handle class imbalance
        n_estimators=300, # Number of boosting rounds: sequential trees, each correcting the last, unlike random forest, the trees are trained independently
        random_state=42,
        n_jobs=-1
    )
}

## Build ML Pipeline

In [0]:
# Pipeline chains steps together so that when called .fit or . predict(), it automatically runs each step in sequence
pipelines = {
    name: Pipeline([
        # For each model, create a pipeline using the same shared preprocessor and that specific model, stored under the model's name as the key
        ('preprocessing', preprocessor),
        ('model', model)
    ])
    for name, model in models.items()
}

In [0]:
# Train each pipeline on labeled data — models learn the relationship
# between input features (X_train) and known outcomes (y_train)

for name, pipeline in pipelines.items(): # Loops through all three trained pipelines again, same as before.

    pipeline.fit(X_train, y_train) 
    print(f"{name} trained successully")

### Metrics

In [0]:
for name, pipeline in pipelines.items():
    y_pred = pipeline.predict(X_test) # Runs X_test through the pipeline to get predicted values for each lead
   
    y_prob = pipeline.predict_proba(X_test)[:, 1] # Predicts the probability of each lead converting (1) or not converting (0)
   
   # Compares y_pred against actual oucomes y_test to calculate evaluation metrics
    print(classification_report(y_test, y_pred))
    print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

### Threshold Tunning

In [0]:

# Select Logistic Regression as the best model and gets the conversion probablity for each lead in the test set
best_pipeline = pipelines['Logistic Regression']
y_prob = best_pipeline.predict_proba(X_test)[:, 1]

# Calculates precision and recall at every possible thresholf from 0 to 1
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)

# Find the threshold with best F1
f1_scores = 2 * (precisions * recalls) / (precisions + recalls) # Calculates the F1 score at every threshold using the F1 formula
best_threshold = thresholds[np.argmax(f1_scores)] # np.argmax(f1_scores) returns the index of the highest F1 score, then uses that index to pull the corresponding threshold value: ie the point where precision and recall are best balance

print(f"Optimal threshold: {best_threshold:.2f}")
print(f"Precision at optimal: {precisions[np.argmax(f1_scores)]:.2f}")
print(f"Recall at optimal:    {recalls[np.argmax(f1_scores)]:.2f}")

# Apply the new threshold
y_pred_tuned = (y_prob >= best_threshold).astype(int)

### Hyperparameter Tuning

In [0]:
# The parameter grid
param_grid = {
    'model__C': [0.001, 0.01, 0.1, 1, 10, 100], # C controls regularization strength. Smaller vales = stronger regularization (simpler model, less overfitting). Larger values = weaker regularization (more complex model, risks overfitting)

    'model__penalty': ['l1', 'l2', 'elasticnet'], # Penalty is the type of regularization applied. L1 = Lasso(Can shrink some feature weights into exactly zero), L2 = Ridge( shrink all weights but rarely to zero. default, more common), ElasticNet = combination of both

    'model__solver': ['saga'], # Solver is the algorithm used to optimize the model. 'saga' is the only solver that supports all three types of penalties

    'model__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9] # only relevant when penalty is elasticnet. Controls the mix between l1 and l2 (0 = pure l2, 1 = pure l1)
}

# RandomizedSearchCV
search = RandomizedSearchCV(
    pipelines['Logistic Regression'],
    param_distributions=param_grid,
    n_iter=50, # Test 50 random combinations from the grid instead of all possible ones.
    scoring='roc_auc', # Uses ROC-AUC to judge which parameter combination is best, consistent with how you evaluated models earlier
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    random_state=42, # for each combination, splits the training data into 5 folds, trains on 4 and validates on 1, rotating through all 5. The final score is the average across all 5 folds — much more reliable than a single split.
    #shuffle=True — shuffles data before splitting into folds to avoid order bias
    n_jobs=-1 # runs combinations in parallel using all CPU cores
)

# Fitting and results
search.fit(X_train, y_train)
print(f"Best ROC-AUC: {search.best_score_:.4f}")
print(f"Best params:  {search.best_params_}")

In [0]:
best_model = search.best_estimator_
y_prob = best_model.predict_proba(X_test)[:, 1]

# Re-run threshold tuning on the new model
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls)
best_threshold = thresholds[np.argmax(f1_scores)]
print(f"New optimal threshold: {best_threshold:.2f}")

# Apply the new optimal threshold
y_pred_tuned = (y_prob >= best_threshold).astype(int)

print(classification_report(y_test, y_pred_tuned))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")
print(confusion_matrix(y_test, y_pred_tuned))

In [0]:
results_df = pd.DataFrame({
    'conversion_probability': y_prob,
    'actual_converted': y_test.values
})

# Assign tiers
def assign_tier(prob):
    if prob >= 0.70:
        return 'High Priority'
    elif prob >= 0.50:
        return 'Medium Priority'
    else:
        return 'Low Priority'

results_df['tier'] = results_df['conversion_probability'].apply(assign_tier)

# Summarise each tier
tier_summary = results_df.groupby('tier').agg(
    total_leads=('actual_converted', 'count'),
    actual_converters=('actual_converted', 'sum'),
    avg_probability=('conversion_probability', 'mean')
).reset_index()

tier_summary['conversion_rate'] = (tier_summary['actual_converters'] / tier_summary['total_leads'] * 100).round(1)

print(tier_summary)

In [0]:
for threshold in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.61]:
    y_pred_thresh = (y_prob >= threshold).astype(int)
    report = classification_report(y_test, y_pred_thresh, output_dict=True)
    auc_roc = roc_auc_score(y_test, y_prob)
    print(f"\nThreshold: {threshold}")
    print(f"  Class 1 Precision: {report['1']['precision']:.2f}")
    print(f"  Class 1 Recall:    {report['1']['recall']:.2f}")
    print(f"  Class 1 F1:        {report['1']['f1-score']:.2f}")
    print(f"  Accuracy:          {report['accuracy']:.2f}")

In [0]:
# Estimates — adjust these to realistic numbers for your business
revenue_per_conversion = 560      # $ value of each converted lead
cost_per_outreach = 30             # $ cost of sales rep contacting a lead

# At your current threshold (0.61)
y_pred_current = (y_prob >= 0.61).astype(int)

true_positives  = ((y_pred_current == 1) & (y_test == 1)).sum()  # caught converters
false_positives = ((y_pred_current == 1) & (y_test == 0)).sum()  # wasted outreach
false_negatives = ((y_pred_current == 0) & (y_test == 1)).sum()  # missed converters

revenue_captured  = true_positives  * revenue_per_conversion
revenue_missed    = false_negatives * revenue_per_conversion
cost_of_outreach  = (true_positives + false_positives) * cost_per_outreach
net_value         = revenue_captured - cost_of_outreach

print(f"Revenue captured:       ${revenue_captured:,}")
print(f"Revenue missed:         ${revenue_missed:,}")
print(f"Cost of outreach:       ${cost_of_outreach:,}")
print(f"Net value generated:    ${net_value:,}")
print(f"Leads correctly caught: {true_positives} out of {true_positives + false_negatives} actual converters")

In [0]:
# Assumptions
revenue_per_conversion = 1000  # blended average
cost_per_outreach = 75

thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.61, 0.70]

results = []

for threshold in thresholds:
    y_pred_t = (y_prob >= threshold).astype(int)
    
    tp = ((y_pred_t == 1) & (y_test == 1)).sum()  # correctly caught converters
    fp = ((y_pred_t == 1) & (y_test == 0)).sum()  # false alarms
    fn = ((y_pred_t == 0) & (y_test == 1)).sum()  # missed converters
    
    leads_contacted     = tp + fp
    conversions_captured = tp
    outreach_cost       = leads_contacted * cost_per_outreach
    revenue             = conversions_captured * revenue_per_conversion
    net_profit          = revenue - outreach_cost
    
    results.append({
        'threshold':            threshold,
        'leads_contacted':      leads_contacted,
        'conversions_captured': conversions_captured,
        'missed_converters':    fn,
        'outreach_cost':        outreach_cost,
        'revenue':              revenue,
        'net_profit':           net_profit
    })

results_df_thresh = pd.DataFrame(results)
print(results_df_thresh.to_string(index=False))

In [0]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Threshold Simulation — Business Impact', fontsize=16, fontweight='bold')

# Net Profit
axes[0, 0].plot(results_df_thresh['threshold'], results_df_thresh['net_profit'], 
                marker='o', color='green', linewidth=2.5, markersize=8)
axes[0, 0].set_title('Net Profit vs Threshold')
axes[0, 0].set_xlabel('Threshold')
axes[0, 0].set_ylabel('Net Profit ($)')
axes[0, 0].axvline(x=0.61, color='red', linestyle='--', alpha=0.6, label='Current (0.61)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Leads Contacted vs Conversions Captured
axes[0, 1].plot(results_df_thresh['threshold'], results_df_thresh['leads_contacted'],
                marker='o', color='steelblue', linewidth=2.5, markersize=8, label='Leads Contacted')
axes[0, 1].plot(results_df_thresh['threshold'], results_df_thresh['conversions_captured'],
                marker='s', color='orange', linewidth=2.5, markersize=8, label='Conversions Captured')
axes[0, 1].set_title('Leads Contacted vs Conversions Captured')
axes[0, 1].set_xlabel('Threshold')
axes[0, 1].set_ylabel('Count')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Revenue vs Outreach Cost
axes[1, 0].plot(results_df_thresh['threshold'], results_df_thresh['revenue'],
                marker='o', color='green', linewidth=2.5, markersize=8, label='Revenue')
axes[1, 0].plot(results_df_thresh['threshold'], results_df_thresh['outreach_cost'],
                marker='s', color='red', linewidth=2.5, markersize=8, label='Outreach Cost')
axes[1, 0].set_title('Revenue vs Outreach Cost')
axes[1, 0].set_xlabel('Threshold')
axes[1, 0].set_ylabel('Amount ($)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Missed Converters
axes[1, 1].plot(results_df_thresh['threshold'], results_df_thresh['missed_converters'],
                marker='o', color='crimson', linewidth=2.5, markersize=8)
axes[1, 1].set_title('Missed Converters vs Threshold')
axes[1, 1].set_xlabel('Threshold')
axes[1, 1].set_ylabel('Missed Converters')
axes[1, 1].axvline(x=0.61, color='red', linestyle='--', alpha=0.6, label='Current (0.61)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('threshold_simulation.png', dpi=150, bbox_inches='tight')
plt.show()

In [0]:
fig, ax = plt.subplots(figsize=(12, 6))

# Plot net profit line
ax.plot(results_df_thresh['threshold'], results_df_thresh['net_profit'],
        marker='o', color='green', linewidth=2.5, markersize=9, zorder=5, label='Net Profit')

# Shade under the curve
ax.fill_between(results_df_thresh['threshold'], results_df_thresh['net_profit'],
                alpha=0.08, color='green')

# Mark optimal point (0.35)
optimal = results_df_thresh.loc[results_df_thresh['net_profit'].idxmax()]
ax.scatter(optimal['threshold'], optimal['net_profit'],
           color='gold', s=200, zorder=6, edgecolors='darkgreen', linewidth=2)
ax.annotate(f"Optimal Threshold: {optimal['threshold']}\nNet Profit: ${optimal['net_profit']:,.0f}",
            xy=(optimal['threshold'], optimal['net_profit']),
            xytext=(optimal['threshold'] + 0.06, optimal['net_profit'] - 3000),
            fontsize=10, fontweight='bold', color='darkgreen',
            arrowprops=dict(arrowstyle='->', color='darkgreen', lw=1.5))

# Mark current ML threshold (0.61)
current = results_df_thresh[results_df_thresh['threshold'] == 0.61].iloc[0]
ax.axvline(x=0.61, color='red', linestyle='--', linewidth=1.8, alpha=0.7)
ax.scatter(current['threshold'], current['net_profit'],
           color='red', s=150, zorder=6)
ax.annotate(f"ML Optimal (F1): 0.61\nNet Profit: ${current['net_profit']:,.0f}",
            xy=(current['threshold'], current['net_profit']),
            xytext=(current['threshold'] + 0.03, current['net_profit'] - 5500),
            fontsize=10, color='red',
            arrowprops=dict(arrowstyle='->', color='red', lw=1.5))

# Difference annotation
ax.annotate('', xy=(0.35, optimal['net_profit']), xytext=(0.61, current['net_profit']),
            arrowprops=dict(arrowstyle='<->', color='steelblue', lw=1.5))
ax.text(0.46, 60000, f"+${optimal['net_profit'] - current['net_profit']:,.0f} difference",
        fontsize=10, color='steelblue', fontweight='bold')

# Labels and formatting
ax.set_title('Net Profit vs Decision Threshold\nML Optimal ≠ Business Optimal',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Decision Threshold', fontsize=12)
ax.set_ylabel('Net Profit ($)', fontsize=12)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_xticks(results_df_thresh['threshold'])
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('net_profit_vs_threshold.png', dpi=150, bbox_inches='tight')
plt.show()